# Summary

This notebook contains a summary of the entire repository for the BNPL project

In [14]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sys, os, glob
import re
import math
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DateType, DoubleType

In [15]:
sys.path.insert(0, "../scripts")
from spark_setup import get_spark
spark = get_spark()

## Preprocessing

On preprocessing all the data the following records are left in the dataset

In [16]:
# merchant data set
df_merchants = spark.read.parquet('.././data/curated/df_merchants')

df_merchants.printSchema()
df_merchants.count()

root
 |-- merchant_abn: long (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- category: string (nullable = true)
 |-- revenue_band: string (nullable = true)
 |-- take_rate: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- n_transactions: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- avg_fraud_probability: double (nullable = true)



4026

In [17]:
# consumer
df_customers = spark.read.parquet('.././data/curated/df_customers')

df_customers.printSchema()
df_customers.count()

root
 |-- consumer_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- order_datetime: date (nullable = true)
 |-- fraud_probability: double (nullable = true)
 |-- is_same_day_duplicate: boolean (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- gender: string (nullable = true)



34864

In [18]:
# transactions (before join w merchant and consumers)
df_transactions = spark.read.parquet('.././data/curated/transactions')

df_transactions.printSchema()
df_transactions.count()

root
 |-- user_id: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)



14195505

All tables have been checked to ensure that all records are valid. Invalid merchants were removed.

Joining the original data:
The data was joined on order_datetime and ids for merchants and consumers

In [19]:
# transactions (before join w merchant and consumers)
full_transactions = spark.read.parquet('.././data/curated/df_transactions')

full_transactions.printSchema()
full_transactions.count()

root
 |-- merchant_abn: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- consumer_id: long (nullable = true)
 |-- fraud_probability: double (nullable = true)
 |-- is_same_day_duplicate: boolean (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- category: string (nullable = true)
 |-- revenue_band: string (nullable = true)
 |-- take_rate: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- n_transactions: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- avg_fraud_probability: double (nullable = true)



71816

In [25]:
full_transactions.filter(F.isnull('avg_fraud_probability')).count()

65518

In [26]:
full_transactions.filter(F.isnull('fraud_probability')).count()

0

65518 merchant frauds were null and 0 consumer frauds were null and needed to be imputed.

## External datasets:
**Census Data:** This data was combined with our original data using consumer postocde and was used to retrieve features such as household size, family earnings and age.

In [ ]:
#transactions = spark.read.parquet('.././data/curated/transaction_external')
#transactions.show()

In [ ]:
#transactions.printSchema()

## Exploratory Analysis

This section contains a summary on some of the features studied, `tbc`